# 0. Antes de empezar

Definimos aquí algunas funciones y clases que utilizaremos en esta Notebook

In [ ]:
pip install rasterio

In [ ]:
# Funciones de visualización de la notebook anterior:

def escalar_imagen(imagen, p, nodata = None):
  # Calcular los percentiles p y 100-p
  valor_min = np.percentile(imagen[imagen != nodata], p)
  valor_max = np.percentile(imagen[imagen != nodata], 100-p)

  # Truncar la imagen al rango [p1, p99]
  ecualizada = np.clip(imagen, valor_min, valor_max)

  # Normalizar la imagen al rango 0-255.
  ecualizada = (ecualizada - valor_min) / (valor_max - valor_min) * 255

  # Convertir la imagen ecualizada a entero de 8 bits
  ecualizada = np.uint8(ecualizada)

  # Retornar la imagen ecualizada
  return ecualizada

def escalar_imagen_multibanda(imagen, p, nodata = None):
  # Crear un array vacio con la misma estructura que imagen_spot_apilada
  imagen_escalada = np.empty_like(imagen)

  # Definir una variable con el número de canales
  canales = imagen_escalada.shape[-1]

  # Escalar una a una las bandas (usando p)
  for i in range(canales):
    canal = imagen[:, :, i]
    imagen_escalada[:, :, i] = escalar_imagen(canal, p, nodata = nodata)

  # Convertir a np.uint8 el array escalado (no hace falta seguir usando float)
  imagen_escalada = np.uint8(imagen_escalada)

  return imagen_escalada

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab.output import eval_js
from base64 import b64encode
from PIL import Image
import io
from shapely.geometry import box
import rasterio
import rasterio.transform

class Extractor():
    def __init__(self, img_np, img_rgb, metadata) -> None:
        self.extracted_dataframe = pd.DataFrame()
        self.extracted_polygons = gpd.GeoDataFrame()  # GeoDataFrame para polígonos georreferenciados
        self.img_np = img_np
        self.img_rgb = img_rgb
        self.metadata = metadata  # Diccionario de metadatos de rasterio
        self.callback = self.manejar_df_extraido

    def manejar_df_extraido(self, df, polygons):
        self.extracted_dataframe = df.copy()
        self.extracted_polygons = polygons  # Guardar el GeoDataFrame de polígonos extraídos

    def interfaz_seleccion_pixeles(self):
        scale_factor = 2.5  # Factor de escalado (aumenta el tamaño de la imagen)
        img_pil = Image.fromarray(self.img_rgb)
        buffered = io.BytesIO()
        img_pil.save(buffered, format="PNG")
        img_b64 = b64encode(buffered.getvalue()).decode()

        new_width = self.img_rgb.shape[1] * scale_factor
        new_height = self.img_rgb.shape[0] * scale_factor

        js_code = f"""
        <style>
          canvas {{ border: 3px solid black; display: block; margin: auto; }}
        </style>
        <canvas id="canvas" width="{new_width}" height="{new_height}"></canvas>
        <script>
          var scaleFactor = {scale_factor};
          var canvas = document.getElementById('canvas');
          var ctx = canvas.getContext('2d');
          var img = new Image();
          img.src = "data:image/png;base64,{img_b64}";
          img.onload = function() {{
              ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
          }};

          var rects = [];
          var startX, startY, drawing = false;

          canvas.addEventListener('mousedown', function(e) {{
              startX = e.offsetX / scaleFactor;
              startY = e.offsetY / scaleFactor;
              drawing = true;
          }});

          canvas.addEventListener('mouseup', function(e) {{
              if (!drawing) return;
              drawing = false;
              var endX = e.offsetX / scaleFactor;
              var endY = e.offsetY / scaleFactor;
              rects.push([startX, startY, endX, endY]);

              ctx.strokeStyle = 'red';
              ctx.lineWidth = 2;
              ctx.strokeRect(startX * scaleFactor, startY * scaleFactor,
                             (endX - startX) * scaleFactor,
                             (endY - startY) * scaleFactor);
          }});

          function getRects() {{ return rects; }}
          function clearRects() {{
              rects = [];
              ctx.clearRect(0, 0, canvas.width, canvas.height);
              ctx.drawImage(img, 0, 0, canvas.width, canvas.height);
          }}
        </script>
        """
        display(HTML(js_code))

        def get_rectangles():
            return eval_js("getRects()")

        def extract_pixels():
            rects = get_rectangles()
            if not rects:
                print("No rectangles drawn.")
                return pd.DataFrame(), gpd.GeoDataFrame()

            dataframes = []
            extracted_polygons = []
            transform = self.metadata.get("transform")
            crs = self.metadata.get("crs")

            for i, (x1, y1, x2, y2) in enumerate(rects):
                x1, x2 = sorted([int(x1), int(x2)])
                y1, y2 = sorted([int(y1), int(y2)])

                # Convertir a coordenadas georreferenciadas
                lon1, lat1 = rasterio.transform.xy(transform, y1, x1, offset='center')
                lon2, lat2 = rasterio.transform.xy(transform, y2, x2, offset='center')
                polygon = box(lon1, lat1, lon2, lat2)
                extracted_polygons.append(polygon)

                # Extraer valores de todas las bandas
                selected_pixels = self.img_np[:, y1:y2, x1:x2]
                pixel_values = selected_pixels.reshape(selected_pixels.shape[0], -1).T
                df = pd.DataFrame(pixel_values, columns=[f"Band_{b}" for b in range(self.img_np.shape[0])])
                df["X"] = np.tile(np.arange(x1, x2), y2 - y1)
                df["Y"] = np.repeat(np.arange(y1, y2), x2 - x1)
                df["Rect_ID"] = i + 1
                dataframes.append(df)

            # Crear un GeoDataFrame con los polígonos y asignar el CRS
            gdf = gpd.GeoDataFrame(geometry=extracted_polygons, crs=crs)

            return pd.concat(dataframes, ignore_index=True) if dataframes else pd.DataFrame(), gdf

        extract_button = widgets.Button(description="Extraer Valores")
        clear_button = widgets.Button(description="Borrar Todo")
        output = widgets.Output()

        def on_extract_clicked(b):
            with output:
                output.clear_output()
                extracted_df, extracted_polygons = extract_pixels()
                if self.callback:
                    self.manejar_df_extraido(extracted_df, extracted_polygons)

        def on_clear_clicked(b):
            eval_js("clearRects()")

        extract_button.on_click(on_extract_clicked)
        clear_button.on_click(on_clear_clicked)

        display(extract_button, clear_button, output)
        return None


# Notebook 03a: Generación de muestras para la clasificación supervisada

En esta primer notebook vamos a generar los datos que necesitamos para entrenar a un clasificador supervisado. Para ellos nos ayudaremos de la clase `Extractor` que ya venimos usando. Dibujaremos rectángulos sobre las zonas o clases que queremos identificar, e iremos guardando la información en _DataFrames_ de _Pandas_. Finalizaremos guardando esos datos como archivo plano para usarlo en la siguiente Notebook, donde se entrena a los clasificadores y se obtienen los mapas.

### Datos con los que trabajaremos en esta notebook:

- Recorte de la misma Sentinel que se generó en la Clase 02 (`s2_clip.tif`).

```python
import gdown

# ID del archivo de Google Drive
file_id = "1i5OEwoHtuVWLMagae3RjY2jxRqr55Ctl"
url = f"https://drive.google.com/uc?id={file_id}"

# Descargar la imagen
output = "s2_clip.tif"
gdown.download(url, output, quiet=False)
```

## 1. Extracción de muestras

Vamos a usar `Extractor` para generar muestras sobre el recorte de la Sentinel-2 realizado en la Clase 02. Recordá subir al mismo a la sección de archivos antes de correr el código de abajo:

```python
#Leer la imagen s2_clip.tif
s2 = rasterio.open('s2_clip.tif')

#Leer los metadatos
metadatos = s2.meta

# Leer como array de numpy
s2_clip = s2.read()

#Escalar para combinaciones RGB
s2_clip_escalado = escalar_imagen_multibanda(s2_clip.transpose(1,2,0), 1)
```



Ahora vamos a repetir lo realizado en la Notebook 02a, donde extrajimos muestras para agua, vegetación, suelo desnudo y zonas construidas. Hoy vamos a ampliar la cantidad de clases. Vas a tomar muestras de Agua _Limpia_, Agua con posible contenido de clorofila y/o materia orgánica (que le vamos a lamar _Agua Cl_), Suelo Desnudo, Canteras y Vegetación. Ejecutá en **celdas separadas** cada uno de los siguientes códigos:

In [ ]:
# Agua (excluir cuerpo con tonalidad verdosa, que probablemente sea agua con algún otro componente)
seleccion_pixeles_agua = Extractor(s2_clip, s2_clip_escalado[:,:,[4,3,2]], s2.meta)
seleccion_pixeles_agua.interfaz_seleccion_pixeles()

In [ ]:
# Agua con Cl y/o materia orgánica (se ve más verde)
seleccion_pixeles_agua_cl = Extractor(s2_clip, s2_clip_escalado[:,:,[4,3,2]], s2.meta)
seleccion_pixeles_agua_cl.interfaz_seleccion_pixeles()

In [ ]:
# Vegetacion
seleccion_pixeles_vegetacion = Extractor(s2_clip, s2_clip_escalado[:,:,[4,3,2]], s2.meta)
seleccion_pixeles_vegetacion.interfaz_seleccion_pixeles()

In [ ]:
# Suelo desnudo
seleccion_pixeles_suelo = Extractor(s2_clip, s2_clip_escalado[:,:,[4,3,2]], s2.meta)
seleccion_pixeles_suelo.interfaz_seleccion_pixeles()

In [ ]:
# Suelo desnudo (canteras)
seleccion_pixeles_canteras = Extractor(s2_clip, s2_clip_escalado[:,:,[4,3,2]], s2.meta)
seleccion_pixeles_canteras.interfaz_seleccion_pixeles()

In [ ]:
# Zonas construidas
seleccion_pixeles_construido = Extractor(s2_clip, s2_clip_escalado[:,:,[4,3,2]], s2.meta)
seleccion_pixeles_construido.interfaz_seleccion_pixeles()

Una vez que hayas extraído los píxeles, vamos a agregarle una columna a cada uno de ellos que nos sirva para identificar la clase. Asignaremos `0`para el Agua, `1` para el Agua con materia orgánica, `2` para la vegetación, `3` para el suelo desnudo, `4` para las Canteras y `5` para las construcciones. Almacenaremos ese valor en una columna llamada _Clase_:

```python
seleccion_pixeles_agua.extracted_dataframe['Clase'] = 0
seleccion_pixeles_agua_cl.extracted_dataframe['Clase'] = 1
seleccion_pixeles_vegetacion.extracted_dataframe['Clase'] = 2
seleccion_pixeles_suelo.extracted_dataframe['Clase'] = 3
seleccion_pixeles_canteras.extracted_dataframe['Clase'] = 4
seleccion_pixeles_construido.extracted_dataframe['Clase'] = 5
```

Una vez hecho esto, podemos proceder  a unir estos 6 _DataFrames_ en uno solo, usando `concat()` de _Pandas_, que ha sido importado como `pd`:

```python
df_final = pd.concat([seleccion_pixeles_agua.extracted_dataframe,
                      seleccion_pixeles_agua_cl.extracted_dataframe,
                      seleccion_pixeles_vegetacion.extracted_dataframe,
                      seleccion_pixeles_suelo.extracted_dataframe,
                      seleccion_pixeles_canteras.extracted_dataframe,
                      seleccion_pixeles_construido.extracted_dataframe],
                      ignore_index=True)
```

De esta manera, ahora contamos con un _DataFrame_ o tabla que tiene la información de reflectancia de las 6 bandas de la imagen Sentinel para cada una de las clases o coberturas del suelo identificadas. Podemos además agregar una descripción como texto de qué es cada clase. Por ejemplo, si tenemos definido este diccionario:

```python
dict_clases = {0: "Agua", 1:"Agua Cl", 2: "Vegetacion", 3: "Suelo Desnudo", 4:"Canteras", 5: "Construido"}
```

Podemos usar la función `.map()` de _Pandas_ para aplicar ese diccionario sobre la columna `Clase`.

```python
df_final["DESC"] = df_final["Clase"].map(dict_clases)

#Visualizar el dataframe
df_final.head()
```
Así, a todas las filas con el valor `0`les va a cargar el valor `Agua` en la nueva columna `DESC`, a todas las que tienen el valor `1`, va a cargar el valor `Vegetacion`, y así con el resto de las clases.

## 1.1. Visualización de firmas espectrales y gráficos de dispersión

Al igual que lo hicimos en la Notebook 02a, podemos visualizar los patrones espectrales de estas muestras que acabamos de extraer. Sin embargo, el código para hacerlo cambia un poco, dado que ahora todas las muestras están dentro de un mismo _DataFrame_:

```python
import matplotlib.pyplot as plt

# Columnas correspondientes a las reflectancias
band_columns = ["Band_0", "Band_1", "Band_2", "Band_3", "Band_4", "Band_5"]

# Longitudes de onda (en nanómetros) de las bandas Sentinel-2 utilizadas
wavelengths_nm = np.array([490, 560, 665, 842, 1610, 2190])  # Bandas 2, 3, 4, 8, 11, 12

# Calcular promedio y desviación estándar por clase
stats = df_final.groupby("Clase")[band_columns].agg(["mean", "std"])

# Extraer los valores en arrays para cada clase
clases = {0: "Agua", 1:"Agua Cl", 2: "Vegetacion", 3: "Suelo Desnudo", 4:"Canteras", 5: "Construido"}
colores = {0: "blue", 1:"red", 2: "green", 3: "pink", 4:"yellow", 5: "gray"}

plt.figure(figsize=(8, 6))

for clase, nombre in clases.items():
    if clase in stats.index:
        bandas_promedio = stats.loc[clase, (slice(None), "mean")].values
        bandas_std = stats.loc[clase, (slice(None), "std")].values
        plt.plot(wavelengths_nm, bandas_promedio, 'o-', label=nombre, color=colores[clase])
        plt.fill_between(wavelengths_nm, bandas_promedio - bandas_std, bandas_promedio + bandas_std,
                         color=colores[clase], alpha=0.3)

plt.xlabel("Longitud de onda (nm)")
plt.ylabel("Valor promedio de la banda")
plt.title("Patrones espectrales de diferentes clases")
plt.legend()
plt.grid(True)
plt.show()
```

Otro tipo de analisis que nos permite ver la separabilidad de estas clases de manera visual, es realizar gráficos de dispersión en 2D, donde se contrastan las reflectancias de dos bandas diferentes. Por ejemplo, el código de abajo utiliza la función [`scatterplot()` de Seaborn](https://seaborn.pydata.org/generated/seaborn.scatterplot.html) para visualizar las reflectancias de la columna _Band_2_, que corresponde al rojo, contra las reflectancias de la columna _Band_1_, que corresponde al verde:

```python
import seaborn as sns

# Especificamos las columnas que queremos poner en los ejes X e Y:
b_x = "Band_2"
b_y = "Band_1"

colores = {"Agua": "blue", "Agua Cl":"red", "Vegetacion": "green", "Suelo Desnudo": "pink", "Canteras":"yellow", "Construido": "gray"}

# Configurar tamaño de la figura
fig = plt.figure(figsize = (10,10))

# Crear el scatter plot con Seaborn
ax = sns.scatterplot(data=df_final, x=df_final[b_x], y=df_final[b_y], hue="DESC", alpha=0.3, s=15, palette=colores)

# Ajustar los límites del gráfico (opcional)
ax.set_xlim([df_final[b_x].min(), df_final[b_x].max()])
ax.set_ylim([df_final[b_y].min(), df_final[b_y].max()])

# Etiquetas y título
ax.set_xlabel(f"Reflectancia en {b_x}")
ax.set_ylabel(f"Reflectancia en {b_y}")
ax.set_title("Distribución de Píxeles por Clase")

# Mostrar gráfico
plt.show()
```

Podemos usar `seaborn` para mirar rápidamente los histogramas de cada clase y los gráficos de dispersión entre os distintos pares de pandas para darnos una idea de la separabilidad que tenemos de las clases en el espacio de atributos.

```python
from matplotlib import pyplot as plt
import seaborn as sns

columnas = ['Band_0', 'Band_1', 'Band_2', 'Band_3', 'Band_4', 'Band_5','DESC']

plt.figure()
sns.pairplot(df_final[columnas],hue='DESC',palette= colores)
plt.show()
```

## 1.2. Visualización de los polígonos de las muestras junto a la imagen

Otra visualización que podemos realizar es superponer los polígonos extraídos a nuestra imagen, y colorearlos según la clase. Para ello, podemos usar otro método de nuestra clase _Extractor_, que es `.extracted_polygons()`. Este método nos devuelve las geometrías de los rectángulos que marcamos sobre cada una de las zonas. Al igual que concatenamos las muestras de píxeles en un solo _DataFrame_, podemos hacer lo mismo y concatenar los polígonos en un solo _GeoDataFrame_. Antes de hacer eso, agregamos a cada uno una columna `Clase` para poder luego diferenciarlos. Luego, le cargamos el campo `DESC` usando el mismo diccionario de antes.

Luego de concatenarlos, volvermos a convertir el _DataFrame_ generado en _GeoDataFrame_:

```python
# Agregamos la descripción de las clases
seleccion_pixeles_agua.extracted_polygons['Clase'] = 0
seleccion_pixeles_agua_cl.extracted_polygons['Clase'] = 1
seleccion_pixeles_vegetacion.extracted_polygons['Clase'] = 2
seleccion_pixeles_suelo.extracted_polygons['Clase'] = 3
seleccion_pixeles_canteras.extracted_polygons['Clase'] = 4
seleccion_pixeles_construido.extracted_polygons['Clase'] = 5

# Concatemos los polígonos
poligonos_clases = pd.concat([seleccion_pixeles_agua.extracted_polygons,
                      seleccion_pixeles_agua_cl.extracted_polygons,
                      seleccion_pixeles_vegetacion.extracted_polygons,
                      seleccion_pixeles_suelo.extracted_polygons,
                      seleccion_pixeles_canteras.extracted_polygons,
                      seleccion_pixeles_construido.extracted_polygons],
                      ignore_index=True)

# Agregamos la descripción de clase como hicimos con las muestras
dict_clases = {0: "Agua", 1:"Agua Cl", 2: "Vegetacion", 3: "Suelo Desnudo", 4:"Canteras", 5: "Construido"}
poligonos_clases["DESC"] = poligonos_clases["Clase"].map(dict_clases)

# Convertimos de nuevo en GeoDataFrame
poligonos_clases = gpd.GeoDataFrame(poligonos_clases)
```

Con este _GeoDataFrame_ generado, podemos hacer un ploteo similar al hecho en la Clase 02a, donde le pasamos una _extensión_ geográfica a `imshow()` a partir de las coordenadas de la imagen. Teniendo esa extensión definida, podemos superponer los polígonos sin inconvenientes:

```python
left, bottom, right, top = s2.bounds  # Extraer los límites

# Crear la figura y los ejes
fig, ax = plt.subplots(figsize=(10, 10))

# Mostrar la imagen con los límites geoespaciales
ax.imshow(s2_clip_escalado[:, :, [4, 3, 2]], extent=[left, right, bottom, top], origin="upper")
ax.ticklabel_format(style="plain", axis="y")

#Agregar los polígonos
poligonos_clases.plot(ax=ax, facecolor='none', column = 'DESC', linewidth=2, legend = True)

ax.set_title("Imagen con Polígonos Superpuestos")

plt.show()

```

El método `plot()` de `Geopandas` permite colorear los polígonos con el parámetro `column`, y al poner explícitamente `Legend = True` se carga la leyenda con las referencias en la figura

## 1.3. Repetir el proceso y guardar las muestras y los polígonos

Una vez que estés conforme con los datos, podés guardar tanto los polígonos como el `df_final` para usarlo en la siguiente Notebbok donde correremos los clasificadores.

```python
# Guardamos los polígonos como archivo GeoPackage
poligonos_clases.to_file('poligonos_muestras.gpkg')

# Guardamos el dataframe como archivo CSV.
df_final.to_csv('muestras.csv')
```

# 2. Yapa: Extracción de muestras con polígonos externos.

Es posible que a veces contemos ya con nuestros propios vectores. En tal caso no es necesario usar _Extractor()_, sino que podemos utilizar la operación _Mask_ para extraer los recortes de la imagen y convertirlos en un _DataFrame_ de Pandas. Lo importante es que en esos polígonos externos tengamos una columna o atributo que nos permita diferenciarlos por clase. En este caso, hemos preparado un vector con la misma estructura de columnas que tienen nuestras muestras.

## 2.1. Lectura de un poligono de muestras externo

A continuación leeremos el archivo que se encuentra disponible en Google Drive:

```python
import gdown

# ID del archivo de Google Drive
file_id = "13oXy_Dw6WOe1UrKlhJGyS-aM4TeIF5ta"
url = f"https://drive.google.com/uc?id={file_id}"

# Descargar la imagen
output = "poligonos_externos.gpkg"
gdown.download(url, output, quiet=False)

# Importar la librería geopandas
import geopandas as gpd

poligonos_externos = gpd.read_file(output)
poligonos_externos.head()
```

El _GeoDataFrame_ importado tiene una columa `Clase`, cuyos valores únicos se pueden consultar haciendo:

```python
print(poligonos_externos['Clase'].unique())
print(poligonos_externos['DESC'].unique())
```

In [ ]:
print(poligonos_externos['Clase'].unique())
print(poligonos_externos['DESC'].unique())

Si los ploteamos junto a la imagen, verificaremos que se trata de rectangulos al igual que los generados con _Extractor()_

```python
left, bottom, right, top = s2.bounds  # Extraer los límites

# Crear la figura y los ejes
fig, ax = plt.subplots(figsize=(10, 10))

# Mostrar la imagen con los límites geoespaciales
ax.imshow(s2_clip_escalado[:, :, [4, 3, 2]], extent=[left, right, bottom, top], origin="upper")
ax.ticklabel_format(style="plain", axis="y")

# Agregar los polígonos
poligonos_externos.plot(ax=ax, facecolor='none', column = 'DESC', linewidth=2, legend = True)

ax.set_title("Imagen con Polígonos Superpuestos")

plt.show()
```

## 2.2. Generación de muestras con los polígonos

Podemos iterar con el siguiente código sobre los polígonos, ir aplicando el mask con cada uno sobre la imagen, y convertir los `arrays` extraídos a un _DataFrame_:

```python
from rasterio.mask import mask

# Suponiendo que `muestras_extraidas` está vacío al principio
muestras_extraidas = pd.DataFrame()

valor_relleno = 65535

# Iterar sobre cada fila de poligonos_externos
for i, pol in poligonos_externos.iterrows():
    clip, _ = mask(dataset=s2, shapes=[pol.geometry], crop=True, nodata=valor_relleno)
    d, x, y = clip.shape
    clip = clip.reshape([d, x * y]).T
    clip = pd.DataFrame(clip, columns=["Band_0", "Band_1", "Band_2", "Band_3", "Band_4", "Band_5"])

    # Eliminar filas que contienen el valor de relleno
    clip = clip[(clip != valor_relleno).all(axis=1)]

    # Agregar las columnas de Clase y de  ID de polígono
    clip['Clase'] = pol['Clase']
    clip['DESC'] = pol['DESC']
    clip['Rect_ID'] = i + 1

    # Concatenar el DataFrame `clip` con `muestras_extraidas`
    muestras_extraidas = pd.concat([muestras_extraidas, clip], ignore_index=True)
```

Si miramos el contenido de nuestro _DataFrame_ `muestras_extraidas`, veremos que tiene prácticamente la misa estructura de lo que generamos con _Extractor()_:

```python
muestras_extraidas.head()
```

De esa manera, nos queda un _DataFrame_ listo para usar en la siguiente Notebook, el cual podemos guardar como archivo CSV:

```python
muestras_extraidas.to_csv("muestras_poligonos_externos.csv")
```

Se deja una función llamada `extraer_muestras` que sistematiza los pasos relalizados, asumiendo que el archivo vectorial tiene una columna que se llama ``Clase`. La usaremos en la siguiente Notebook:

```python
from rasterio.mask import mask

def extraer_muestras(imagen, poligonos_externos, valor_relleno, columnas = ["Band_0", "Band_1", "Band_2", "Band_3", "Band_4", "Band_5"]):
    muestras_extraidas = pd.DataFrame()
    for i, pol in poligonos_externos.iterrows():
        clip, _ = mask(dataset=imagen, shapes=[pol.geometry], crop=True, nodata=valor_relleno)
        d, x, y = clip.shape
        clip = clip.reshape([d, x * y]).T
        clip = pd.DataFrame(clip, columns=columnas)
        clip = clip[(clip != valor_relleno).all(axis=1)]
        clip['Clase'] = pol['Clase']
        clip['DESC'] = pol['DESC']
        clip['Rect_ID'] = i + 1
        muestras_extraidas = pd.concat([muestras_extraidas, clip], ignore_index=True)
    return muestras_extraidas

#Probar la función
muestras_funcion = extraer_muestras(s2, poligonos_externos, 65535)
```

**Ojo! Esta función espera ciertos nombres de columna en el vector de entrada, si usamos otra estructura, deberiamos modificar la función (o nuestro vector)**